# Vector stores and semantic search



In [17]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np

## Part I: Basic vector store implementation

In [18]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: list = []

    def add_documents(self, documents: list[Document]):
        self.documents.extend(documents)
        texts = [doc.text for doc in documents]
        new_embeddings = self.model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
        self.embeddings.extend(new_embeddings)

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        query_embedding = self.model.encode([query])[0]

        # Similitud coseno manual (sin FAISS ni librerías de vector store)
        results = []
        for doc, emb in zip(self.documents, self.embeddings):
            dot = sum(float(a) * float(b) for a, b in zip(query_embedding, emb))
            norm_q = sum(float(a) ** 2 for a in query_embedding) ** 0.5
            norm_e = sum(float(b) ** 2 for b in emb) ** 0.5
            score = dot / (norm_q * norm_e + 1e-10)
            results.append(SearchResult(score=score, document=doc))

        results.sort(key=lambda r: r.score, reverse=True)
        return results[:top_k]

In [19]:
df = pd.read_csv('/Users/alexgarcia/PycharmProjects/InteligenciaComputacional/data/animal-fun-facts-dataset.csv')

# Eliminar filas sin texto
df = df.dropna(subset=["text"])
df = df[df["text"].str.strip() != ""]

documents = [
    Document(
        text=row["text"],
        metadata={
            "animal_name":    row["animal_name"],
            "source":         row["source"]         if pd.notna(row["source"])         else "",
            "media_link":     row["media_link"]     if pd.notna(row["media_link"])     else "",
            "wikipedia_link": row["wikipedia_link"] if pd.notna(row["wikipedia_link"]) else "",
        }
    )
    for _, row in df.iterrows()
]

print(f"Documentos cargados: {len(documents)}")
print(f"\nEjemplo:")
print(f"  text     : {documents[0].text}")
print(f"  metadata : {documents[0].metadata}")

Documentos cargados: 7731

Ejemplo:
  text     : Aardvarks are sometimes called "ant bears", "earth pigs",
and "cape anteaters"
  metadata : {'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}


In [20]:
model = SentenceTransformer("all-MiniLM-L6-v2")
vs = VectorStore(embedding_model=model)
vs.add_documents(documents)

print(f"Documentos indexados: {len(vs.documents)}")

Batches: 100%|██████████| 242/242 [00:08<00:00, 29.37it/s]

Documentos indexados: 7731


In [21]:
queries = [
    "animals that can survive in the desert without water",
    "marine mammals that use echolocation to hunt",
    "birds that cannot fly but run very fast",
    "venomous animals that are dangerous to humans",
    "animals that sleep during winter hibernation",
]

for query in queries:
    print(f"\nQuery: {query}")
    print("-" * 40)
    results = vs.search(query, top_k=3)
    for i, r in enumerate(results, 1):
        print(f"  [{i}] score  : {r.score:.4f}")
        print(f"       animal : {r.document.metadata['animal_name']}")
        print(f"       texto  : {r.document.text[:180]}...")
        print(f"       source : {r.document.metadata['source']}")
        print(f"       wiki   : {r.document.metadata['wikipedia_link']}")
    print()


Query: animals that can survive in the desert without water
----------------------------------------
  [1] score  : 0.7237
       animal : naked mole-rat
       texto  : They survive without drinking water. .
Instead, they extract liquid from the plants they eat. In the dry, desert underground environment, they randomly dig to find roots and tubers...
       source : https://factanimal.com/naked-mole-rat/
       wiki   : /wiki/Naked_mole-rat
  [2] score  : 0.7088
       animal : fennec fox
       texto  : The fennec fox appears to be the only carnivore in the Sahara Desert able to live without freely available water. Their kidneys are specifically adapted to conserve water.  They ca...
       source : https://seaworld.org/animals/facts/mammals/fennec-fox/
       wiki   : /wiki/Fennec_fox
  [3] score  : 0.6936
       animal : baboon
       texto  : Some species can go days without water.
The chacma baboon lives in drier habitats that resemble desert environments....
       source : htt

## Part II: Filtering by metadata

In [22]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: list = []

    def add_documents(self, documents: list[Document]):
        self.documents.extend(documents)
        texts = [doc.text for doc in documents]
        new_embeddings = self.model.encode(texts, show_progress_bar=True)
        self.embeddings.extend(new_embeddings)

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:

        if metadata_filter:
            filtered = [
                (doc, emb)
                for doc, emb in zip(self.documents, self.embeddings)
                if all(
                    doc.metadata.get(key) == value
                    for key, value in metadata_filter.items()
                )
            ]
        else:
            filtered = list(zip(self.documents, self.embeddings))

        if not filtered:
            print("  ⚠️  No hay documentos que coincidan con el filtro.")
            return []

        query_embedding = self.model.encode([query])[0]

        results = []
        for doc, emb in filtered:
            dot = sum(float(a) * float(b) for a, b in zip(query_embedding, emb))
            norm_q = sum(float(a) ** 2 for a in query_embedding) ** 0.5
            norm_e = sum(float(b) ** 2 for b in emb) ** 0.5
            score = dot / (norm_q * norm_e + 1e-10)
            results.append(SearchResult(score=score, document=doc))

        results.sort(key=lambda r: r.score, reverse=True)
        return results[:top_k]

In [23]:
# Carga diferente con ayuda de ChatGPT porque estaba teniendo problemas ocn el csv original
df_news = pd.read_csv("/Users/alexgarcia/PycharmProjects/InteligenciaComputacional/data/bbc-news-data.csv",sep="\t",on_bad_lines="skip")
df_news = df_news.dropna(subset=["content"])

news_documents = [
    Document(
        text=row["content"],
        metadata={
            "category": row["category"],
            "title":    row["title"],
            "filename": row["filename"],
        }
    )
    for _, row in df_news.iterrows()
]

print(f"Documentos cargados: {len(news_documents)}")
print(f"\nCategorías disponibles: {df_news['category'].unique().tolist()}")
print(f"\nEjemplo:")
print(f"  text     : {news_documents[0].text[:100]}...")
print(f"  metadata : {news_documents[0].metadata}")

Documentos cargados: 2225

Categorías disponibles: ['business', 'entertainment', 'politics', 'sport', 'tech']

Ejemplo:
  text     :  Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months t...
  metadata : {'category': 'business', 'title': 'Ad sales boost Time Warner profit', 'filename': '001.txt'}


In [24]:
fvs = FilteredVectorStore(embedding_model=model)
fvs.add_documents(news_documents)

print(f"Documentos indexados: {len(fvs.documents)}")

Batches: 100%|██████████| 70/70 [00:13<00:00,  5.38it/s]

Documentos indexados: 2225


In [25]:
filtered_queries = [
    {
        "query": "stock market crash and economic recession",
        "filter": {"category": "business"}
    },
    {
        "query": "prime minister election campaign political party",
        "filter": {"category": "politics"}
    },
    {
        "query": "Champions League football tournament results",
        "filter": {"category": "sport"}
    },
    {
        "query": "new smartphone technology artificial intelligence",
        "filter": {"category": "tech"}
    },
    {
        "query": "box office movie awards film festival",
        "filter": {"category": "entertainment"}
    },
]

for item in filtered_queries:
    print(f"\nQuery  : {item['query']}")
    print(f"Filtro : {item['filter']}")
    print("-" * 40)
    results = fvs.search(item["query"], top_k=3, metadata_filter=item["filter"])
    for i, r in enumerate(results, 1):
        print(f"  [{i}] score    : {r.score:.4f}")
        print(f"       titulo   : {r.document.metadata['title']}")
        print(f"       categoria: {r.document.metadata['category']}")
        print(f"       texto    : {r.document.text[:150]}...")
    print()


Query  : stock market crash and economic recession
Filtro : {'category': 'business'}
----------------------------------------
  [1] score    : 0.4877
       titulo   : Japan narrowly escapes recession
       categoria: business
       texto    :  Japan's economy teetered on the brink of a technical recession in the three months to September, figures show.  Revised figures indicated growth of j...
  [2] score    : 0.4711
       titulo   : Wall Street cheers Bush victory
       categoria: business
       texto    :  The US stock market has closed higher in response to George W Bush's victory in the presidential elections.  The benchmark Dow Jones share index clos...
  [3] score    : 0.4677
       titulo   : Japan economy slides to recession
       categoria: business
       texto    :  The Japanese economy has officially gone back into recession for the fourth time in a decade.  Gross domestic product fell by 0.1% in the last three ...


Query  : prime minister election campaign politic

## Reflexión personal

En esta actividad entendí de forma más clara cómo funcionan los embeddings, básicamente 
convierten texto en números, pero lo interesante es que no solo toman en cuenta las palabras 
exactas sino también el significado general de la oración

También vi que filtrar por metadatos antes de calcular la similitud sirve para acotar la 
búsqueda sin necesidad de cambiar el modelo, en el caso del BBC News dataset, restringir 
por categoría hizo que los resultados fueran notablemente más relevantes que buscando sobre 
todos los documentos

Por último, noté que mientras más documentos se indexan, más lenta se vuelve la búsqueda 
porque el sistema compara la query contra cada documento uno por uno. Esto deja claro por 
qué en producción se usan índices optimizados en lugar de búsqueda exhaustiva